In [1]:
from pathlib import Path
import re
import datetime

# ===== Experiment setup =====
MODEL = "llama3.3-70b"
TEMPERATURE = 0
CONFIG_NAME = "C-Full"

scenario_name = "Batmobile functional"

iteration_id = 0

def slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

# ===== Create log file =====
logs_dir = Path("logs")
logs_dir.mkdir(exist_ok=True)




In [2]:
logs_dir

PosixPath('logs')

In [3]:
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
#txt_path = logs_dir / f"{slugify(scenario_name)}__{slugify(CONFIG_NAME)}__{stamp}.txt"
txt_path = logs_dir /  f"batmobile_functional__c_full__20260310-140930.txt"

"""with txt_path.open("w", encoding="utf-8") as f:
    f.write("EXPERIMENT LOG\n")
    f.write("=" * 70 + "\n")
    f.write(f"SCENARIO: {scenario_name}\n")

    f.write(f"CONFIG: {CONFIG_NAME}\n")
    f.write(f"MODEL: {MODEL}\n")
    f.write(f"TEMPERATURE: {TEMPERATURE}\n")
    f.write(f"CREATED_AT: {stamp}\n")
    f.write("=" * 70 + "\n\n")
"""
print("Log file created:", txt_path)

Log file created: logs/batmobile_functional__c_full__20260310-140930.txt


In [4]:
def dict_to_bullets(d, indent: int = 4) -> str:
    """
    Turn a nested dict (and lists) into indented bullet lines.

    Change vs previous version:
    - FIRST-LEVEL keys are NOT printed (e.g., 'arch', 'relation', 'cara' won't appear).
    - Content under each first-level key is printed directly.
    """
    pad = " " * indent
    lines = []

    if d is None or d == {}:
        return f"{pad}- (none)"

    def walk(node, level: int, show_key: bool = True):
        pad2 = " " * level

        if isinstance(node, dict):
            if not node:
                lines.append(f"{pad2}- (none)")
                return
            for k, v in node.items():
                # Skip printing the first-level key labels
                if show_key:
                    key = str(k) if k is not None and str(k).strip() else "(unnamed)"
                    lines.append(f"{pad2}- {key}:")
                    walk(v, level + 2, show_key=True)
                else:
                    # First level: don't print key, just descend into its value
                    walk(v, level, show_key=True)

        elif isinstance(node, (list, tuple, set)):
            node = list(node)
            if not node:
                lines.append(f"{pad2}- (none)")
            else:
                for item in node:
                    if isinstance(item, (dict, list, tuple, set)):
                        lines.append(f"{pad2}-")
                        walk(item, level + 2, show_key=True)
                    else:
                        lines.append(f"{pad2}- {item}")

        else:
            val = str(node).strip()
            lines.append(f"{pad2}- {val if val else '(empty)'}")

    # Start at root with show_key=False so first-level keys are hidden
    walk(d, indent, show_key=False)
    return "\n".join(lines)


In [8]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local-dummy")

def generate_completion(
    scenario_name: str,
    elements: dict,
    model: str,
    temperature: float = 0.0,
    n_additions: int = 3,
    dsl: str = "UML class diagram",
    ontology_annotation: str = "",
    metamodel_spec: str = "",
) -> tuple[str, str]:
    

    # Build partial model text from elements
    elems_block =  dict_to_bullets(elements, indent=4)
    
    partial_model = f"""Elements: \n {elems_block}"""

    prompt = f"""
You are a modeling assistant. Complete a partial {dsl} for the scenario.

Scenario: {scenario_name}

Ontology annotations (domain concepts to use, with meanings):
{ontology_annotation}

Metamodel (allowed element types, relations, and attributes — follow strictly):
{metamodel_spec}

Current partial model:
{partial_model}

Task:
Propose up to {n_additions} additions to extend the model. An addition could be:
- type: ARCH (new class) or CONN (new association), or CHAR (new attribute/constraint/property)

Constraints:
- suggest typed by connections
- inspire from the existing elements and relations to maintain coherence. 
- Do not  suggest Transmission as partdefinition or speedlimit or Battery or Steering or StabilityRequirement. Maybe try with PartUsage that typed to existing PartDefinition
- Only propose additions that are valid according to the metamodel.
- Do not repeat elements that already exist in the partial model.

Write as a bullet list. Be concise.
""".strip()


    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    output_text = resp.choices[0].message.content
    return prompt, output_text






In [9]:
iteration_id=41

In [23]:
iteration_id += 1
ontology_annotation = """Use these mappings:
    - Element, NamedElement, Usage, PartUsage -> Thing
    - Type, Definition, Typing -> NaturalKind
    - Feature, PortDefinition, PortUsage -> Property
    - Membership -> Container
    - PartDefinition -> Item
    - RequirementDefinition, RequirementUsage, ConstraintDefinition, ConstraintUsage, constrains -> Goal
    - ConnectsTo -> Interaction
    - DependsOn -> Process
    - DerivesFrom -> Activity
"""
metamodel_spec = metamodel_spec = """Allowed SysMLv2 elements and relations:

Core elements: (you can use these to build the model, mainly definitions and usages)
- Package(name)
- PartDefinition(name)
- PartUsage(name)
- PortDefinition(name)
- PortUsage(name)
- RequirementDefinition(name)
- RequirementUsage(name)
- ConstraintDefinition(name)
- ConstraintUsage(name)



Allowed relations:
- DependsOn(source:<PartUsage>, target:<PartUsage>)
- ConnectsTo(source:<PortUsage>, target:<PortUsage>)
- DerivesFrom(source:<RequirementUsage>, target:<RequirementUsage>)
- constrains(source:<ConstraintUsage>, target:<PartUsage>)

Generalization constraints:
- Definition is a Type
- Usage is a Feature
- PartDefinition, PortDefinition, RequirementDefinition, ConstraintDefinition are Definitions
- PartUsage, PortUsage, RequirementUsage, ConstraintUsage are Usages

Modeling rules:
- Use only the elements and relations listed above.
- New architectural and characteristics additions should be SysMLv2 concepts such as PartDefinition, PartUsage, PortDefinition, PortUsage, RequirementDefinition, RequirementUsage, ConstraintDefinition, or ConstraintUsage.
- regarding connections suggest per run only one Connection addition that  must use exact metamodel relation names: DependsOn, ConnectsTo, DerivesFrom, constrains, Membership, or Typing.
- Keep suggestions consistent with the current partial model.
- """
prompt, output_text = generate_completion(
    scenario_name=scenario_name,
    elements = {
        "arch": {"partDefinition": {"Engine", "Chassis","Wheels","Sensor", "armor", "Panel", "Axle","Navigation","Frame", "Module","Body", "Component","element"}, 
                "portDefinition": {"PowerPort","FuelPort","ControlPort", "DataPort", "Asset"},
                 "requirementDefinition":{ "HighSpeed"},

                },
        "relation": {       
             "connection- PartDefintion-isTypedBy-PartUsage": {"Engine-Engine", "Wheels-Wheels","Chassis-Chassis","Panel-Panel","Axle-Axle","component-Component","armor-armor"},
            "connection- PortUsage-DependsOn- PortUsage  ": {"axle-engine "},
             "connection": {"sensors-ConnectsTo-DataPort"},


           },
         
             
        
        "characteristics": { 
                 
                  'RequirementUsage  ': {'HighSpeedRequirement', 'CrewSafety',"NavigationAccurate"},
                   'PartUsage  ': {'chassis',"engine","wheels","sensors","Panel", "armor","Frame", "Navigation","axle","Body", "Module"}  ,
                   "ConstraintUsage": {"WeaponsOnlyWhenAuthorized", "EngineRequiresFuel"}               },

            
        

    },
    model=MODEL,
    temperature=TEMPERATURE,
    n_additions=3,
    dsl="sysmlv2 ",
    ontology_annotation=ontology_annotation,
    metamodel_spec=metamodel_spec,
)

print("===== PROMPT =====")
print(prompt)
print("\n===== MODEL OUTPUT =====")
print(output_text)


===== PROMPT =====
You are a modeling assistant. Complete a partial sysmlv2  for the scenario.

Scenario: Batmobile functional

Ontology annotations (domain concepts to use, with meanings):
Use these mappings:
    - Element, NamedElement, Usage, PartUsage -> Thing
    - Type, Definition, Typing -> NaturalKind
    - Feature, PortDefinition, PortUsage -> Property
    - Membership -> Container
    - PartDefinition -> Item
    - RequirementDefinition, RequirementUsage, ConstraintDefinition, ConstraintUsage, constrains -> Goal
    - ConnectsTo -> Interaction
    - DependsOn -> Process
    - DerivesFrom -> Activity


Metamodel (allowed element types, relations, and attributes — follow strictly):
Allowed SysMLv2 elements and relations:

Core elements: (you can use these to build the model, mainly definitions and usages)
- Package(name)
- PartDefinition(name)
- PartUsage(name)
- PortDefinition(name)
- PortUsage(name)
- RequirementDefinition(name)
- RequirementUsage(name)
- ConstraintDefinition

In [24]:
# ===== Manually fill this dict (we copy/paste items from output) =====
classified = {
    "arch": [
        {"PartUsage": "Turbocharger"}
    ],
    "connections": [
        {"DependsOn ": "engine-DependsOn-FuelPort"}
        ]
        
    ,
    "characteristics": [
            {"ConstraintUsage": "EngineTemperatureLimit"}


    ],
}


# ===== Log everything (prompt + output + youand the  manual dict we accepted to integrate) =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write(f"\n\nITERATION {iteration_id}\n")
    f.write("-" * 80 + "\n")



    f.write("PROMPT\n")
    f.write("-" * 80 + "\n")
    f.write(prompt + "\n\n")

    f.write("MODEL OUTPUT\n")
    f.write("-" * 80 + "\n")
    f.write(output_text.strip() + "\n\n")

    f.write("MANUAL CLASSIFICATION\n")
    f.write("-" * 80 + "\n")
    f.write("ARCH:\n")
    for x in classified["arch"]:
        f.write(f"- {x}\n")
    f.write("\nCONNECTIONS:\n")
    for x in classified["connections"]:
        f.write(f"- {x}\n")
    f.write("\nCHARACTERISTICS:\n")
    for x in classified["characteristics"]:
        f.write(f"- {x}\n")

print(f"\nLogged iteration {iteration_id} to:", txt_path)
classified


Logged iteration 51 to: logs/batmobile_functional__c_full__20260310-140930.txt


{'arch': [{'PartUsage': 'Turbocharger'}],
 'connections': [{'DependsOn ': 'engine-DependsOn-FuelPort'}],
 'characteristics': [{'ConstraintUsage': 'EngineTemperatureLimit'}]}

In [25]:
# ===== Manually fill accepted items for this iteration =====
iteration_id += 1
accepted = {
    "arch": [
       
    ],
    "connections": [
    ],
    "characteristics": [
       
    ],
}

# ===== Append accepted to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nACCEPTED (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in accepted["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in accepted["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in accepted["characteristics"]:
        f.write(f"- {x}\n")

print(f"Accepted suggestions appended for iteration {iteration_id} to:", txt_path)
accepted


Accepted suggestions appended for iteration 52 to: logs/batmobile_functional__c_full__20260310-140930.txt


{'arch': [], 'connections': [], 'characteristics': []}

In [ ]:

print("===== ELEMENTS (BULLET VIEW, TOP-LEVEL KEYS HIDDEN) =====")
#"Panel-Panel","Axle-Axle","component-Component","armor-armor"
iteration_id += 1

added_elements ={
    "arch": [
        

    ],
    "connections": [
            {}
    ],
    "characteristics": [
       
    ]
}
# ===== Append new added line  to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nADDED ELEMENTS MANUALLY (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in added_elements["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in added_elements["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in added_elements["characteristics"]:
        f.write(f"- {x}\n")
   

===== ELEMENTS (BULLET VIEW, TOP-LEVEL KEYS HIDDEN) =====
